In [60]:
import os
import certifi
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch as TavilySearchResults
from langsmith import Client

client = Client()

from langchain.agents import create_agent 
from langchain_classic.agents import AgentExecutor

In [61]:
os.environ['SSL_CERT_FILE'] = certifi.where() # this line is necessary to avoid SSL certificate verification errors when making HTTPS requests in some environments, especially when using libraries that rely on SSL/TLS for secure communication. It sets the environment variable 'SSL_CERT_FILE' to the path of the CA bundle provided by the certifi package, which contains a collection of trusted root certificates. This ensures that the application can verify the authenticity of SSL certificates presented by servers during HTTPS requests, preventing potential security issues related to untrusted or invalid certificates.
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL", "nvidia/nemotron-3.5-lightning:free"
)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

Set up Tavily search tool for internet search

In [62]:
search_tool = TavilySearchResults(max_results=3)

In [63]:
#search_tool.invoke("What is the capital of France?")

Initialize LLM

In [64]:
llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    timeout=30,
    max_retries=0,
)

In [65]:
'''
try:
    response = llm.invoke("What color is a sunflower? Reply in one sentence.")
    print(response.content)
except Exception as e:
    print(f"OpenRouter request failed: {type(e).__name__}: {e}")
'''

'\ntry:\n    response = llm.invoke("What color is a sunflower? Reply in one sentence.")\n    print(response.content)\nexcept Exception as e:\n    print(f"OpenRouter request failed: {type(e).__name__}: {e}")\n'

In [66]:
prompt = client.pull_prompt(
    "hwchase17/react", dangerously_pull_public_prompt=True
) # dangerously_pull_public_prompt=True allows pulling public prompts without verification. Use with caution.

In [67]:
tools = [search_tool]

In [68]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the search tool when current information is needed. Always answer in caveman language.",
    debug=True,
) #debug=True enables detailed logging of the agent's actions and decisions, which is useful for debugging and understanding its behavior.

In [76]:
'''
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the capital of France?"}]}
)
response = result["messages"][-1]
print(response.content)
'''

'\nresult = agent.invoke(\n    {"messages": [{"role": "user", "content": "What is the capital of France?"}]}\n)\nresponse = result["messages"][-1]\nprint(response.content)\n'